# 06 - 手写 Qwen3-MoE

> 配套笔记：[01_02 混合专家 MoE](https://github.com/Zoey-Cheng/MLSys-Learning-Notes/blob/main/notes/01_模型基础/01_02_MoE.md) ｜ 知乎：[link](https://zhuanlan.zhihu.com/p/2039331127815557840)

MoE 部分用笔记里的实现。Qwen3-MoE 不等于最基础的 MoE：相对 §2.2 BasicMoE 它有两点改动——(1) 细粒度专家，配置层面（专家数多、`moe_intermediate_size` 小、top_k 大），block 代码不改；(2) `norm_topk_prob` 开关，block 里唯一的代码增量。所以这里 block 代码 = doc §5.2，相对 BasicMoE 只多这个开关；细粒度靠超参体现，本 demo 用小号配置（8 experts / top-2），没拉到真实细粒度规模。MHA 笔记没展开，这里按 Qwen3 风格补一份，让 demo 跑成完整一层。三块都对照 transformers 4.51.3 源码核过（见末尾差异 cell）。

顺序：

1. MoE block — = doc §5.2 / §2.2 + `norm_topk_prob` 开关，routing + dispatch 与 `Qwen3MoeSparseMoeBlock` 一致
2. aux loss — = doc §2.4 Switch，数值与 HF `load_balancing_loss_func` 一致
3. Qwen3 风格 MHA — 笔记未展开，这里补，与 `Qwen3MoeAttention` 数学一致
4. 组装并跑一遍
5. 用 HF Qwen3-MoE 作标准答案，逐元素对齐

## 0. 环境

pin `transformers==4.51.3`。新版 transformers 把 Qwen3-MoE 重构成 `Qwen3MoeTopKRouter` + fused `Qwen3MoeExperts`（3D 权重 + grouped matmul），数学不变，但 expert 的 module layout 和 state_dict 变了，跨版本直接拷权重对不上。4.51.3 是经典实现（`gate=nn.Linear` + `experts=ModuleList`），对应 doc §5.2，可与笔记逐元素对齐。

In [1]:
!nvidia-smi

Sun May 17 04:05:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q "transformers==4.51.3" torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.5 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. MoE block

= doc §5.2，把 §2.1 的 SwiGLU expert 一并拼进来凑成可跑的 class，无改动。routing 与 `Qwen3MoeSparseMoeBlock` 逐行一致：全 N 上 softmax → topk → `norm_topk_prob` renorm → `.to(dtype)`。dispatch 用经典 per-expert `expert_mask` + `index_add_`，新版 HF 把这步融成 fused kernel，数学相同（见 §0）。

In [4]:
class SwiGLU(nn.Module):                       # = doc §2.1 BasicExpert
    def __init__(self, dim, hidden):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden, bias=False)
        self.up_proj   = nn.Linear(dim, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, dim, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class MoEBlock(nn.Module):                      # = doc §5.2 Qwen3MoeSparseMoeBlock（经典版）
    def __init__(self, dim, num_experts, top_k, moe_hidden, norm_topk_prob=True):
        super().__init__()
        self.num_experts, self.top_k, self.norm_topk_prob = num_experts, top_k, norm_topk_prob
        self.gate = nn.Linear(dim, num_experts, bias=False)
        self.experts = nn.ModuleList([SwiGLU(dim, moe_hidden) for _ in range(num_experts)])
    def forward(self, x):                       # x: [B, T, d]
        B, T, d = x.shape
        x = x.reshape(-1, d)
        router_logits = self.gate(x)
        w = F.softmax(router_logits, dim=-1, dtype=torch.float)
        w, sel = torch.topk(w, self.top_k, dim=-1)
        if self.norm_topk_prob:
            w = w / w.sum(-1, keepdim=True)
        w = w.to(x.dtype)
        out = torch.zeros_like(x)
        mask = F.one_hot(sel, self.num_experts).permute(2, 1, 0)
        for e in range(self.num_experts):
            kth, tok = torch.where(mask[e])
            if tok.numel() == 0:
                continue
            out.index_add_(0, tok, self.experts[e](x[tok]) * w[tok, kth, None])
        return out.reshape(B, T, d), router_logits

## 2. aux loss

= doc §2.4 Switch：$L_{aux}=\alpha\cdot N\cdot\sum_i f_i P_i$。$f_i$ 实际分配比例（离散、detach），
$P_i$ 平均 router 概率（可导）。值和 HF `load_balancing_loss_func` 一致（§5 会对齐）。

In [ ]:
def switch_aux_loss(router_logits, top_k, num_experts, coef=0.01):
    probs = F.softmax(router_logits, dim=-1)
    P = probs.mean(0)                                  # [N] 平均概率，可导
    sel = probs.topk(top_k, dim=-1).indices            # 由 logits 复原 top-k
    f = F.one_hot(sel, num_experts).float().mean(0)     # [top_k, N] 实际比例
    return coef * num_experts * (f.detach() * P.unsqueeze(0)).sum()
    # 注：f 保留成 [top_k, N]（= HF load_balancing_loss_func 的 tokens_per_expert，
    #     方便 §5b 逐元素对齐）；doc §2.4 的写法是 .sum(1) 先把 top_k 收掉成 [N]。
    #     Σ_{k,i} f[k,i]·P[i] = Σ_i (Σ_k f[k,i])·P[i] —— 两种 reduction 数值完全相同。

## 3. Qwen3 风格的 MHA

笔记没展开这部分：§5.1 只点了 Qwen3 attention 的改动（q_norm/k_norm、RoPE 动态频率、GQA），MHA 代码在 01-Transformer 那篇。这里补全，方便组装成完整一层。与 `Qwen3MoeAttention` 数学一致：q_norm/k_norm 对 `head_dim` 做 RMSNorm、`rotate_half`、`q*cos + rotate_half(q)*sin`、`repeat_kv`、`scaling = head_dim**-0.5`。

In [ ]:
class RMSNorm(nn.Module):                 # = Qwen3MoeRMSNorm
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        x = x.float()
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (self.weight * x).type_as(self.weight)
        # 注：HF 是 self.weight * x.to(input_dtype)（先 cast 回输入 dtype 再乘 weight）；
        #     这里是 fp32 下乘完 weight 再 cast。本 demo 全程 fp32 → 完全一致；
        #     仅 bf16/fp16 混精时两者最后一步的舍入次序略有不同（不影响本对齐）。

def rope_cos_sin(T, hd, base=10000.0):    # base=10000.0 = Qwen3MoeConfig 默认 rope_theta
    inv = 1.0 / (base ** (torch.arange(0, hd, 2).float() / hd))
    freqs = torch.outer(torch.arange(T).float(), inv)        # [T, hd/2]
    emb = torch.cat([freqs, freqs], dim=-1)                  # [T, hd]
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    cos, sin = cos[None, None], sin[None, None]              # unsqueeze 到 [1,1,T,hd]
    return q*cos + rotate_half(q)*sin, k*cos + rotate_half(k)*sin

def repeat_kv(x, n):                                          # GQA: [B,nkv,T,hd]->[B,nkv*n,T,hd]
    B, nkv, T, hd = x.shape
    return x[:, :, None].expand(B, nkv, n, T, hd).reshape(B, nkv*n, T, hd)

class Qwen3Attention(nn.Module):
    def __init__(self, dim, n_heads, n_kv, head_dim, eps=1e-6):
        super().__init__()
        self.nh, self.nkv, self.hd = n_heads, n_kv, head_dim
        self.q_proj = nn.Linear(dim, n_heads*head_dim, bias=False)   # Qwen3 无 qkv bias
        self.k_proj = nn.Linear(dim, n_kv*head_dim,    bias=False)
        self.v_proj = nn.Linear(dim, n_kv*head_dim,    bias=False)
        self.o_proj = nn.Linear(n_heads*head_dim, dim, bias=False)
        self.q_norm = RMSNorm(head_dim, eps)      # ← Qwen3 特有（doc §5.1）
        self.k_norm = RMSNorm(head_dim, eps)      # ← Qwen3 特有（doc §5.1）
        self.scale = head_dim ** -0.5
    def forward(self, x):
        B, T, _ = x.shape
        q = self.q_norm(self.q_proj(x).view(B, T, self.nh,  self.hd)).transpose(1, 2)
        k = self.k_norm(self.k_proj(x).view(B, T, self.nkv, self.hd)).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        cos, sin = rope_cos_sin(T, self.hd)
        q, k = apply_rope(q, k, cos, sin)
        k, v = repeat_kv(k, self.nh//self.nkv), repeat_kv(v, self.nh//self.nkv)
        o = F.scaled_dot_product_attention(q, k, v, is_causal=True, scale=self.scale)
        return self.o_proj(o.transpose(1, 2).reshape(B, T, -1))

## 4. 组装 & 跑一遍

拼成一层 Qwen3-MoE DecoderBlock（Pre-Norm + Attn + 残差 → Pre-Norm + MoE + 残差），随机输入跑前向。


In [7]:
class Qwen3MoEBlock(nn.Module):
    def __init__(self, dim, n_heads, n_kv, head_dim, num_experts, top_k, moe_hidden):
        super().__init__()
        self.n1 = RMSNorm(dim); self.attn = Qwen3Attention(dim, n_heads, n_kv, head_dim)
        self.n2 = RMSNorm(dim); self.moe  = MoEBlock(dim, num_experts, top_k, moe_hidden)
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        h, router_logits = self.moe(self.n2(x))
        return x + h, router_logits

dim, n_heads, n_kv, head_dim = 128, 4, 2, 32
num_experts, top_k, moe_hidden = 8, 2, 128
blk = Qwen3MoEBlock(dim, n_heads, n_kv, head_dim, num_experts, top_k, moe_hidden).eval()
x = torch.randn(2, 16, dim)
with torch.no_grad():
    y, rl = blk(x)
print('out :', tuple(y.shape))
print('aux :', switch_aux_loss(rl, top_k, num_experts).item())

out : (2, 16, 128)
aux : 0.021349448710680008


## 5. 用 HF Qwen3-MoE 作标准答案，逐元素对齐

4.51.3 的 Qwen3-MoE 是经典 layout（`mlp.gate` = Linear，`mlp.experts` = ModuleList），与 `MoEBlock` 一一对应。拷权重 → 同输入 → 断言 `allclose`。

In [8]:
from transformers import Qwen3MoeConfig, Qwen3MoeForCausalLM
from transformers.models.qwen3_moe.modeling_qwen3_moe import load_balancing_loss_func

cfg = Qwen3MoeConfig(
    hidden_size=dim, intermediate_size=512, moe_intermediate_size=moe_hidden,
    num_hidden_layers=1, num_attention_heads=n_heads, num_key_value_heads=n_kv,
    num_experts=num_experts, num_experts_per_tok=top_k, norm_topk_prob=True,
    decoder_sparse_step=1, router_aux_loss_coef=0.01,
    vocab_size=1000, max_position_embeddings=128, rope_theta=10000.0,
)
# 4.51.3 的 Qwen3MoeConfig 没有 head_dim 字段 → HF 回退到 hidden_size//n_heads = 32，
# 正好等于手写的 head_dim；下面 assert 兜底。rope_theta 也显式写成默认 10000.0 与手写对齐。
cfg._attn_implementation = "sdpa"   # ← 关键：sdpa 在 attention_mask=None & T>1 时自动 is_causal=True，
                                    #    和手写的 SDPA(is_causal=True) 一致；eager 路径不会自动加 causal，
                                    #    那样参考侧会变成全注意力、和手写对不上。
ref = Qwen3MoeForCausalLM(cfg).eval()
L = ref.model.layers[0]
assert isinstance(L.mlp.experts, nn.ModuleList), \
    'experts 不是 ModuleList —— 你的 transformers 版本已重构成 fused Experts，请按 §0 pin 4.51.3'
assert L.self_attn.head_dim == head_dim, f'head_dim 对不上: HF={L.self_attn.head_dim} vs 手写={head_dim}'

# --- 5a. MoE block ---（gate + per-expert，和 doc §5.2 / v4.51.3 Qwen3MoeSparseMoeBlock 逐行一致）
blk.moe.gate.load_state_dict(L.mlp.gate.state_dict())
for e in range(num_experts):
    blk.moe.experts[e].gate_proj.load_state_dict(L.mlp.experts[e].gate_proj.state_dict())
    blk.moe.experts[e].up_proj.load_state_dict(L.mlp.experts[e].up_proj.state_dict())
    blk.moe.experts[e].down_proj.load_state_dict(L.mlp.experts[e].down_proj.state_dict())
h = torch.randn(2, 16, dim)
with torch.no_grad():
    om, lm = blk.moe(h)
    ref_out = L.mlp(h)
    orf, lrf = ref_out if isinstance(ref_out, tuple) else (ref_out, blk.moe.gate(h.reshape(-1,dim)))
print('MoE  diff:', (om - orf).abs().max().item())
assert torch.allclose(om, orf, atol=1e-5)

# --- 5b. aux loss ---
# switch_aux_loss 的系数是 α·N；HF load_balancing_loss_func 是 ×N（α 由模型外层另乘）。
# 两边同除 N → 比较裸 Σ f·P，与 α 无关。
mine = switch_aux_loss(lrf, top_k, num_experts, coef=1.0) / num_experts
hf   = load_balancing_loss_func((lrf,), num_experts, top_k) / num_experts
print('aux  diff:', (mine - hf).abs().item())
assert torch.allclose(mine, hf, atol=1e-6)

# --- 5c. MHA ---（参考侧用 §5 顶部 pin 的 sdpa；返回值取 [0] = attn_output）
for n in ['q_proj','k_proj','v_proj','o_proj','q_norm','k_norm']:
    getattr(blk.attn, n).load_state_dict(getattr(L.self_attn, n).state_dict())
pos = torch.arange(16).unsqueeze(0)
cos, sin = ref.model.rotary_emb(h, pos)          # 同一套 RoPE(rope_theta=10000.0)
with torch.no_grad():
    o_ref  = L.self_attn(h, position_embeddings=(cos, sin), attention_mask=None)[0]
    o_mine = blk.attn(h)
print('MHA  diff:', (o_mine - o_ref).abs().max().item())
assert torch.allclose(o_mine, o_ref, atol=1e-4)
print('三块均与 transformers 4.51.3 标准实现一致')

MoE  diff: 0.0
aux  diff: 0.0
MHA  diff: 0.0
三块均与 transformers 4.51.3 标准实现一致


## 注：与真实 Qwen3 (transformers 4.51.3) / 笔记的差异

逐行比对 `modeling_qwen3_moe.py`：

- MoE block：routing + per-expert dispatch 与 `Qwen3MoeSparseMoeBlock` 逐行一致，也与 doc §5.2 一致（2D 下 `softmax(dim=1)` 等于 `dim=-1`，`/=` 等价 `/`）。SwiGLU = `Qwen3MoeMLP`（silu、无 bias）。
- MHA：q_norm/k_norm、`rotate_half`、`repeat_kv`、`scaling=head_dim**-0.5` 与 `Qwen3MoeAttention` 数学一致。4.51.3 config 无 `head_dim` 字段，回退 `hidden_size//n_heads=32`，与手写相同（已 assert）。RoPE 用默认 `rope_theta=10000.0`，与手写 `base=10000.0` 相同。
- aux loss：与 `load_balancing_loss_func`、doc §2.4 数值一致。

差异都在写法/形状层面，数值相同，已在对应 cell 注释：

1. `switch_aux_loss` 把 `f` 留成 `[top_k, N]` 对齐 HF `tokens_per_expert`；doc §2.4 用 `.sum(1)` 收成 `[N]`。`Σ f·P` 两种 reduction 恒等。
2. `RMSNorm` 末步：本实现 fp32 下乘 weight 再 cast，HF 先 cast 回 input dtype 再乘。fp32（本对齐路径）一致，仅 bf16/fp16 下舍入次序不同。
3. expert layout：本 notebook 与 doc §5.2 用经典 `ModuleList` + `index_add_`（= 4.51.3）；新版 HF 融成 fused `Qwen3MoeExperts`（3D 权重 + grouped matmul），数学等价、kernel 不同。§0 已 pin 4.51.3。
4. §5c 参考侧 pin `attn_implementation="sdpa"`：`attention_mask=None` 且 `T>1` 时 sdpa 自动 `is_causal=True`，与手写一致；eager 不自动加 causal，会变成全注意力。